# XAI: ProtoPNet

## 1. Imports

In [ ]:
import json
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision import transforms
from torchvision.transforms import functional as TF

from sklearn.metrics import accuracy_score, f1_score




## 2. Shared Utilities (Unified Interface)

In [ ]:
# Fix random seeds for reproducible experiments.
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# Automatically locate the project root (must contain `dataset/archive` and `outputs`).
def find_project_root(start: Path) -> Path:
    cur = start.resolve()
    for p in [cur] + list(cur.parents):
        if (p / 'dataset' / 'archive').exists() and (p / 'outputs').exists():
            return p
    raise FileNotFoundError(' Graduation Thesis ')


class ResizeLongestSidePadSquare:
    def __init__(self, size: int, fill: int = 0):
        self.size = int(size)
        self.fill = int(fill)

    def __call__(self, img: Image.Image):
        w, h = img.size
        scale = self.size / max(w, h)
        nw = max(1, int(round(w * scale)))
        nh = max(1, int(round(h * scale)))
        img = TF.resize(img, [nh, nw], interpolation=transforms.InterpolationMode.BILINEAR)
        pw, ph = self.size - nw, self.size - nh
        left, top = pw // 2, ph // 2
        right, bottom = pw - left, ph - top
        return TF.pad(img, [left, top, right, bottom], fill=self.fill)


class ManifestDataset(Dataset):
    # Build samples from the split manifest and return `(image, label, rel_path)`.
    def __init__(self, dataset_root: Path, df: pd.DataFrame, class_to_idx: dict, transform=None):
        self.dataset_root = Path(dataset_root)
        self.transform = transform
        self.class_to_idx = dict(class_to_idx)
        self.samples = []
        for _, r in df.iterrows():
            rel = str(r['path'])
            cls = str(r['class'])
            if cls not in self.class_to_idx:
                continue
            p = self.dataset_root / rel
            if p.exists():
                self.samples.append((p, self.class_to_idx[cls], rel))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        p, y, rel = self.samples[i]
        x = Image.open(p).convert('RGB')
        if self.transform is not None:
            x = self.transform(x)
        return x, y, rel


def normalize_map(m: np.ndarray) -> np.ndarray:
    m = m.astype(np.float32)
    m -= m.min()
    d = m.max() + 1e-8
    return m / d


def topk_mask(score_map: np.ndarray, ratio: float = 0.1) -> np.ndarray:
    h, w = score_map.shape
    flat = score_map.reshape(-1)
    k = max(1, int(round(len(flat) * ratio)))
    idx = np.argpartition(flat, -k)[-k:]
    mask = np.zeros_like(flat, dtype=np.float32)
    mask[idx] = 1.0
    return mask.reshape(h, w)


def iou(a: np.ndarray, b: np.ndarray) -> float:
    inter = np.logical_and(a > 0, b > 0).sum()
    union = np.logical_or(a > 0, b > 0).sum()
    return float(inter / (union + 1e-8))


def target_margin_from_logits(logits: torch.Tensor, cls_idx: int) -> torch.Tensor:
    target = logits[:, int(cls_idx)]
    other = logits.clone()
    other[:, int(cls_idx)] = -1e9
    other_max = other.max(dim=1).values
    return target - other_max


def model_target_score(model, x: torch.Tensor, cls_idx: int, score_type: str = 'margin') -> float:
    with torch.no_grad():
        logits = model(x)
        if score_type == 'margin':
            return float(target_margin_from_logits(logits, cls_idx).item())
        if score_type == 'logit':
            return float(logits[0, int(cls_idx)].item())
        if score_type == 'prob':
            return float(torch.softmax(logits, dim=1)[0, int(cls_idx)].item())
    raise ValueError(f'unsupported score_type: {score_type}')


def make_replacement_baseline(x: torch.Tensor, mode: str = 'blur', blur_ks: int = 31) -> torch.Tensor:
    if mode == 'zero':
        return torch.zeros_like(x)
    if mode == 'mean':
        return x.mean(dim=(2, 3), keepdim=True).expand_as(x)
    if mode == 'blur':
        k = int(max(3, blur_ks))
        if k % 2 == 0:
            k += 1
        return torch.nn.functional.avg_pool2d(x, kernel_size=k, stride=1, padding=k // 2)
    raise ValueError(f'unsupported replace mode: {mode}')


def compose_with_keep_mask(x: torch.Tensor, keep_mask: torch.Tensor, mode: str = 'blur', blur_ks: int = 31) -> torch.Tensor:
    base = make_replacement_baseline(x, mode=mode, blur_ks=blur_ks)
    return x * keep_mask + base * (1.0 - keep_mask)


def deletion_aopc(model, x: torch.Tensor, cls_idx: int, score_map: np.ndarray, steps=5, replace_mode='blur', score_type='margin', blur_ks=31) -> float:
    # Deletion-AOPC: progressively remove high-importance regions and measure the drop in the target score.
    s0 = model_target_score(model, x, cls_idx, score_type=score_type)

    h, w = score_map.shape
    imp = score_map.reshape(-1)
    order = np.argsort(-imp)
    drops = []
    total = h * w

    for s in range(1, steps + 1):
        cut = int(total * s / steps)
        idx = order[:cut]
        keep = np.ones(total, dtype=np.float32)
        keep[idx] = 0.0
        keep = torch.from_numpy(keep.reshape(1, 1, h, w)).to(x.device)
        if x.shape[-2:] != (h, w):
            keep = torch.nn.functional.interpolate(keep, size=x.shape[-2:], mode='nearest')
        x_mask = compose_with_keep_mask(x, keep, mode=replace_mode, blur_ks=blur_ks)
        s_del = model_target_score(model, x_mask, cls_idx, score_type=score_type)
        drops.append(s0 - s_del)

    return float(np.mean(drops))


def insertion_auc(model, x: torch.Tensor, cls_idx: int, score_map: np.ndarray, steps=5, replace_mode='blur', score_type='margin', blur_ks=31) -> float:
    # Insertion-AUC: progressively insert high-importance regions and compute the area under the target-score curve.
    h, w = score_map.shape
    imp = score_map.reshape(-1)
    order = np.argsort(-imp)
    total = h * w
    xs = []
    ys = []

    for s in range(1, steps + 1):
        keep_n = int(total * s / steps)
        idx = order[:keep_n]
        keep = np.zeros(total, dtype=np.float32)
        keep[idx] = 1.0
        keep = torch.from_numpy(keep.reshape(1, 1, h, w)).to(x.device)
        if x.shape[-2:] != (h, w):
            keep = torch.nn.functional.interpolate(keep, size=x.shape[-2:], mode='nearest')
        x_ins = compose_with_keep_mask(x, keep, mode=replace_mode, blur_ks=blur_ks)
        s_ins = model_target_score(model, x_ins, cls_idx, score_type=score_type)
        xs.append(s / steps)
        ys.append(s_ins)

    return float(np.trapezoid(ys, xs))



def comprehensiveness(model, x: torch.Tensor, cls_idx: int, score_map: np.ndarray, top_ratio=0.25, replace_mode='blur', score_type='margin', blur_ks=31) -> float:
    # Faithfulness: drop in the target score after removing important regions; higher is better.
    h, w = score_map.shape
    m = topk_mask(score_map, top_ratio)
    m = torch.from_numpy(m).float().view(1, 1, h, w).to(x.device)
    if x.shape[-2:] != (h, w):
        m = torch.nn.functional.interpolate(m, size=x.shape[-2:], mode='nearest')

    s_full = model_target_score(model, x, cls_idx, score_type=score_type)
    x_drop = compose_with_keep_mask(x, 1.0 - m, mode=replace_mode, blur_ks=blur_ks)
    s_drop = model_target_score(model, x_drop, cls_idx, score_type=score_type)
    return float(s_full - s_drop)


def sufficiency(model, x: torch.Tensor, cls_idx: int, score_map: np.ndarray, top_ratio=0.25, replace_mode='blur', score_type='margin', blur_ks=31) -> float:
    # Faithfulness: target-score loss when only important regions are kept; lower is better.
    h, w = score_map.shape
    m = topk_mask(score_map, top_ratio)
    m = torch.from_numpy(m).float().view(1, 1, h, w).to(x.device)
    if x.shape[-2:] != (h, w):
        m = torch.nn.functional.interpolate(m, size=x.shape[-2:], mode='nearest')

    s_full = model_target_score(model, x, cls_idx, score_type=score_type)
    x_keep = compose_with_keep_mask(x, m, mode=replace_mode, blur_ks=blur_ks)
    s_keep = model_target_score(model, x_keep, cls_idx, score_type=score_type)
    return float(s_full - s_keep)


def selected_feature_ratio(score_map: np.ndarray, threshold: float = 0.5) -> float:
    # Complexity: selected-feature ratio; lower means sparser explanations.
    m = normalize_map(score_map)
    return float((m >= threshold).mean())


def sparsity_from_ratio(ratio: float) -> float:
    # Complexity: sparsity = 1 - selected-feature ratio; higher means sparser explanations.
    return float(1.0 - ratio)



def save_explanation_visual(x: torch.Tensor, score_map: np.ndarray, out_png: Path, mean, std):
    # Save a visualization that overlays the explanation heatmap on the input image.
    img = x.detach().cpu()[0].permute(1, 2, 0).numpy()
    mean = np.array(mean, dtype=np.float32).reshape(1, 1, 3)
    std = np.array(std, dtype=np.float32).reshape(1, 1, 3)
    img = np.clip(img * std + mean, 0.0, 1.0)
    h, w = img.shape[:2]

    hm = normalize_map(score_map)
    hm = np.array(Image.fromarray((hm * 255).astype(np.uint8)).resize((w, h), Image.BILINEAR)).astype(np.float32) / 255.0

    heat = np.zeros((h, w, 3), dtype=np.float32)
    heat[..., 0] = hm
    heat[..., 2] = 1.0 - hm

    overlay = np.clip(0.65 * img + 0.35 * heat, 0.0, 1.0)

    out_png.parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray((overlay * 255).astype(np.uint8)).save(out_png)

def stability_iou(explain_fn, model, x: torch.Tensor, cls_idx: int, top_ratio=0.1, noise_std=0.20) -> float:
    base_map, _ = explain_fn(model, x, cls_idx)
    n = torch.clamp(x + torch.randn_like(x) * noise_std, -5, 5)
    new_map, _ = explain_fn(model, n, cls_idx)
    a = topk_mask(base_map, top_ratio)
    b = topk_mask(new_map, top_ratio)
    return iou(a, b)





## 3. Global Configuration

In [ ]:
PROJECT_ROOT = find_project_root(Path.cwd())
DATASET_ROOT = PROJECT_ROOT / 'dataset'
SPLIT_DIR = PROJECT_ROOT / 'outputs' / 'data_prep'
SPLIT_ALL_PATH = SPLIT_DIR / 'split_all.csv'
PREPROCESS_YAML = SPLIT_DIR / 'preprocess_config.yaml'
PREPROCESS_JSON = SPLIT_DIR / 'preprocess_config.json'

BASELINE_KEY = 'swin_t'
MAX_SAMPLES = 0
SEED = 42
NUM_WORKERS = 0
USE_PRETRAINED_BACKBONE = True
FORCE_RETRAIN = False

# Align training hyper-parameters with baseline notebooks
BASELINE_TRAIN_CFG = {
    'resnet18': {'batch_size': 128, 'epochs': 20, 'lr': 1e-3, 'weight_decay': 1e-4},
    'vgg16_bn': {'batch_size': 128, 'epochs': 20, 'lr': 1e-3, 'weight_decay': 1e-4},
    'vit_b_16': {'batch_size': 32, 'epochs': 20, 'lr': 1e-4, 'weight_decay': 0.05},
    'swin_t': {'batch_size': 32, 'epochs': 20, 'lr': 5e-5, 'weight_decay': 0.05},
}
BASE_CFG = BASELINE_TRAIN_CFG.get(BASELINE_KEY, BASELINE_TRAIN_CFG['swin_t'])
BATCH_SIZE = int(BASE_CFG['batch_size'])
PROTOPNET_EPOCHS = int(BASE_CFG['epochs'])
PROTOPNET_LR = float(BASE_CFG['lr'])
PROTOPNET_WEIGHT_DECAY = float(BASE_CFG['weight_decay'])

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
set_seed(SEED)

if PREPROCESS_YAML.exists():
    try:
        import yaml
        preprocess_cfg = yaml.safe_load(PREPROCESS_YAML.read_text(encoding='utf-8'))
    except Exception:
        preprocess_cfg = json.loads(PREPROCESS_JSON.read_text(encoding='utf-8')) if PREPROCESS_JSON.exists() else {}
elif PREPROCESS_JSON.exists():
    preprocess_cfg = json.loads(PREPROCESS_JSON.read_text(encoding='utf-8'))
else:
    preprocess_cfg = {}

IMG_SIZE = int(preprocess_cfg.get('img_size', 224))
PAD_FILL = int(preprocess_cfg.get('pad_fill', 0))
NORM_MEAN = tuple(preprocess_cfg.get('normalize', {}).get('mean', [0.485, 0.456, 0.406]))
NORM_STD = tuple(preprocess_cfg.get('normalize', {}).get('std', [0.229, 0.224, 0.225]))
AUG = preprocess_cfg.get('train_augment', {})
FLIP_P = float(AUG.get('random_horizontal_flip', 0.5))
CJ = AUG.get('color_jitter', {'brightness': 0.2, 'contrast': 0.2, 'saturation': 0.2, 'hue': 0.02})

split_df = pd.read_csv(SPLIT_ALL_PATH)
classes = sorted(split_df['class'].dropna().unique().tolist())
class_to_idx = {c: i for i, c in enumerate(classes)}

train_df = split_df[split_df['split'] == 'train'].copy().reset_index(drop=True)
val_df = split_df[split_df['split'] == 'val'].copy().reset_index(drop=True)
test_df = split_df[split_df['split'] == 'test'].copy().reset_index(drop=True)
if MAX_SAMPLES > 0:
    test_df = test_df.iloc[:MAX_SAMPLES].copy().reset_index(drop=True)

base_resize_pad = ResizeLongestSidePadSquare(IMG_SIZE, fill=PAD_FILL)
train_tfms = transforms.Compose([
    base_resize_pad,
    transforms.RandomHorizontalFlip(p=FLIP_P),
    transforms.ColorJitter(
        brightness=float(CJ.get('brightness', 0.2)),
        contrast=float(CJ.get('contrast', 0.2)),
        saturation=float(CJ.get('saturation', 0.2)),
        hue=float(CJ.get('hue', 0.02)),
    ),
    transforms.ToTensor(),
    transforms.Normalize(NORM_MEAN, NORM_STD),
] )

eval_tfms = transforms.Compose([
    base_resize_pad,
    transforms.ToTensor(),
    transforms.Normalize(NORM_MEAN, NORM_STD),
])

train_ds = ManifestDataset(DATASET_ROOT, train_df, class_to_idx, transform=train_tfms)
val_ds = ManifestDataset(DATASET_ROOT, val_df, class_to_idx, transform=eval_tfms)
test_ds = ManifestDataset(DATASET_ROOT, test_df, class_to_idx, transform=eval_tfms)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=NUM_WORKERS)

print('[INFO] baseline key =', BASELINE_KEY)
print('[INFO] proto cfg: batch=', BATCH_SIZE, 'epochs=', PROTOPNET_EPOCHS, 'lr=', PROTOPNET_LR, 'wd=', PROTOPNET_WEIGHT_DECAY)
print('[INFO] train/val/test =', len(train_ds), len(val_ds), len(test_ds))


## 4. Method Implementation

In [ ]:
METHOD_KEY = 'protopnet'
VARIANT_KEY = 'intrinsic'
OUT_DIR = PROJECT_ROOT / 'outputs' / 'xai' / 'intrinsic' / METHOD_KEY / BASELINE_KEY
PROTOPNET_CKPT = OUT_DIR / 'protopnet_best.pt'

PROTO_PER_CLASS = 8
PROTO_EXPLAIN_TOPM = 6
PROTO_SCORE_PLOW = 20.0
PROTO_SCORE_PHIGH = 99
PROTO_SCORE_GAMMA = 1.5
PROTO_NEG_ALPHA = 0.7
PROTO_TEMP = 1.0
LAMBDA_CLUSTER = 0.8
LAMBDA_SEP = 0.08
LAMBDA_L1 = 1e-4
SEP_MARGIN = 1.0

# Faithfulness evaluation settings
EVAL_REPLACE_MODE = 'blur'  # Replacement baseline: blur / mean / zero
EVAL_SCORE_TYPE = 'prob'  # Score type used for AOPC / AUC / Sufficiency
EVAL_BLUR_KS = 31
EVAL_COMP_TOP_RATIO = 0.25
EVAL_SUFF_TOP_RATIO = 0.40

class SwinProtoPNet(nn.Module):
    def __init__(self, num_classes: int, proto_per_class: int = 8, use_pretrained: bool = True, temp: float = 1.0):
        super().__init__()
        self.num_classes = int(num_classes)
        self.proto_per_class = int(proto_per_class)
        self.num_prototypes = self.num_classes * self.proto_per_class
        self.temp = float(temp)

        weights = torchvision.models.Swin_T_Weights.IMAGENET1K_V1 if use_pretrained else None
        backbone = torchvision.models.swin_t(weights=weights)
        self.features = backbone.features
        self.norm = backbone.norm
        self.permute = backbone.permute

        self.feat_dim = 768
        self.prototype_vectors = nn.Parameter(torch.randn(self.num_prototypes, self.feat_dim) * 0.1)
        self.last_layer = nn.Linear(self.num_prototypes, self.num_classes, bias=False)

        p2c = []
        for c in range(self.num_classes):
            p2c.extend([c] * self.proto_per_class)
        self.register_buffer('proto_class_ids', torch.tensor(p2c, dtype=torch.long))

        with torch.no_grad():
            w = torch.full((self.num_classes, self.num_prototypes), -0.5)
            for j in range(self.num_prototypes):
                w[int(self.proto_class_ids[j]), j] = 1.0
            self.last_layer.weight.copy_(w)

    def backbone_features(self, x: torch.Tensor) -> torch.Tensor:
        z = self.features(x)
        z = self.norm(z)
        z = self.permute(z)
        return z

    def _dist_maps(self, feat: torch.Tensor) -> torch.Tensor:
        b, d, h, w = feat.shape
        x = feat.permute(0, 2, 3, 1).reshape(b, h * w, d)
        p = self.prototype_vectors.unsqueeze(0).expand(b, -1, -1)
        dist = torch.cdist(x, p, p=2).pow(2)
        dist = dist.permute(0, 2, 1).reshape(b, self.num_prototypes, h, w)
        return dist

    def forward(self, x: torch.Tensor, return_maps: bool = False):
        feat = self.backbone_features(x)
        dist_maps = self._dist_maps(feat)
        min_dist = dist_maps.flatten(2).min(dim=2).values
        proto_act = torch.exp(-min_dist / self.temp)
        logits = self.last_layer(proto_act)
        if return_maps:
            proto_act_maps = torch.exp(-dist_maps / self.temp)
            return logits, proto_act_maps, min_dist
        return logits


def proto_losses(model: SwinProtoPNet, y: torch.Tensor, min_dist: torch.Tensor):
    pcls = model.proto_class_ids.to(y.device)
    mask_pos = (pcls.unsqueeze(0) == y.unsqueeze(1))
    big = torch.full_like(min_dist, 1e6)

    min_pos = torch.where(mask_pos, min_dist, big).min(dim=1).values
    min_neg = torch.where(~mask_pos, min_dist, big).min(dim=1).values

    cluster = min_pos.mean()
    separation = torch.relu(SEP_MARGIN - min_neg).mean()

    c = model.num_classes
    proto_onehot = torch.nn.functional.one_hot(model.proto_class_ids, num_classes=c).T.float().to(y.device)
    l1 = (torch.abs(model.last_layer.weight) * (1.0 - proto_onehot)).mean()
    return cluster, separation, l1


@torch.no_grad()
def eval_proto(model: SwinProtoPNet, loader):
    model.eval()
    ys, ps = [], []
    for x, y, _ in loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)
        logits = model(x)
        pred = logits.argmax(1)
        ys.extend(y.cpu().numpy().tolist())
        ps.extend(pred.cpu().numpy().tolist())
    acc = accuracy_score(ys, ps) if len(ys) else 0.0
    f1 = f1_score(ys, ps, average='macro') if len(ys) else 0.0
    return acc, f1


OUT_DIR.mkdir(parents=True, exist_ok=True)
protopnet = SwinProtoPNet(len(classes), proto_per_class=PROTO_PER_CLASS, use_pretrained=USE_PRETRAINED_BACKBONE, temp=PROTO_TEMP).to(DEVICE)

if PROTOPNET_CKPT.exists() and not FORCE_RETRAIN:
    payload = torch.load(PROTOPNET_CKPT, map_location=DEVICE)
    protopnet.load_state_dict(payload['model_state_dict'])
    print('[INFO] loaded ProtoPNet checkpoint:', PROTOPNET_CKPT)
else:
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.AdamW(protopnet.parameters(), lr=PROTOPNET_LR, weight_decay=PROTOPNET_WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=PROTOPNET_EPOCHS)

    best_f1 = -1.0
    best_state = None

    for ep in range(1, PROTOPNET_EPOCHS + 1):
        protopnet.train()
        losses = []
        for x, y, _ in train_loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)

            logits, _, min_dist = protopnet(x, return_maps=True)
            ce = criterion(logits, y)
            cluster, sep, l1 = proto_losses(protopnet, y, min_dist)
            loss = ce + LAMBDA_CLUSTER * cluster + LAMBDA_SEP * sep + LAMBDA_L1 * l1

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(protopnet.parameters(), max_norm=1.0)
            optimizer.step()
            losses.append(loss.item())

        scheduler.step()
        val_acc, val_f1 = eval_proto(protopnet, val_loader)
        print(f'[ProtoPNet][{ep:02d}/{PROTOPNET_EPOCHS}] loss={np.mean(losses):.4f} val_acc={val_acc:.4f} val_f1={val_f1:.4f}')

        if val_f1 > best_f1:
            best_f1 = val_f1
            best_state = {k: v.detach().cpu() for k, v in protopnet.state_dict().items()}

    if best_state is not None:
        protopnet.load_state_dict(best_state)

    torch.save({
        'model_state_dict': protopnet.state_dict(),
        'base_cfg': BASE_CFG,
        'proto_per_class': PROTO_PER_CLASS,
        'temp': PROTO_TEMP,
        'best_val_f1': float(best_f1),
    }, PROTOPNET_CKPT)
    print('[OK] saved ProtoPNet checkpoint:', PROTOPNET_CKPT)


@torch.no_grad()
def explain_fn(model: SwinProtoPNet, x: torch.Tensor, cls_idx: int):
    t0 = time.time()
    _, proto_act_maps, _ = model(x, return_maps=True)

    # Weight prototype contributions toward the target class and suppress prototypes favored by other classes.
    w_all = model.last_layer.weight
    w_pos = torch.relu(w_all[int(cls_idx)])
    w_neg = torch.relu(torch.max(torch.cat([w_all[:int(cls_idx)], w_all[int(cls_idx)+1:]], dim=0), dim=0).values)

    amax = proto_act_maps[0].flatten(1).max(dim=1).values
    contrib = torch.relu(w_pos - PROTO_NEG_ALPHA * w_neg) * amax
    m = min(int(PROTO_EXPLAIN_TOPM), int((contrib > 0).sum().item()))

    if m <= 0:
        score = proto_act_maps[0].mean(dim=0)
    else:
        idx = torch.topk(contrib, k=m, dim=0).indices
        sub_maps = proto_act_maps[0][idx]
        sub_w = contrib[idx].view(-1, 1, 1)
        score = (sub_maps * sub_w).sum(dim=0) / (sub_w.sum() + 1e-8)

    score = normalize_map(score.detach().cpu().numpy())
    # Apply percentile rescaling and gamma sharpening to make the explanation map clearer.
    lo = float(np.percentile(score, PROTO_SCORE_PLOW))
    hi = float(np.percentile(score, PROTO_SCORE_PHIGH))
    if hi > lo + 1e-8:
        score = np.clip((score - lo) / (hi - lo + 1e-8), 0.0, 1.0)
    score = np.power(score, PROTO_SCORE_GAMMA)
    score = normalize_map(score)
    return score, time.time() - t0

baseline_model = protopnet


## 5. Unified Quantitative Evaluation and Outputs

In [ ]:
VIS_SAVE_N = 20
VIS_DIR = OUT_DIR / 'visuals'
VIS_DIR.mkdir(parents=True, exist_ok=True)

records = []
for i, (x, y, rel) in enumerate(test_loader):
    sample_t0 = time.time()
    x = x.to(DEVICE)
    y = y.to(DEVICE)

    with torch.no_grad():
        logits = baseline_model(x)
        pred = int(logits.argmax(1).item())

    score_map, t_explain = explain_fn(baseline_model, x, pred)
    if i < VIS_SAVE_N:
        stem = Path(rel[0]).stem
        vis_path = VIS_DIR / f"{i:04d}_{stem}.png"
        save_explanation_visual(x, score_map, vis_path, NORM_MEAN, NORM_STD)

    aopc = deletion_aopc(baseline_model, x, pred, score_map, replace_mode=EVAL_REPLACE_MODE, score_type=EVAL_SCORE_TYPE, blur_ks=EVAL_BLUR_KS)
    auci = insertion_auc(baseline_model, x, pred, score_map, replace_mode=EVAL_REPLACE_MODE, score_type=EVAL_SCORE_TYPE, blur_ks=EVAL_BLUR_KS)
    siou = stability_iou(explain_fn, baseline_model, x, pred)
    comp = comprehensiveness(baseline_model, x, pred, score_map, top_ratio=EVAL_COMP_TOP_RATIO, replace_mode=EVAL_REPLACE_MODE, score_type=EVAL_SCORE_TYPE, blur_ks=EVAL_BLUR_KS)
    suff = sufficiency(baseline_model, x, pred, score_map, top_ratio=EVAL_SUFF_TOP_RATIO, replace_mode=EVAL_REPLACE_MODE, score_type=EVAL_SCORE_TYPE, blur_ks=EVAL_BLUR_KS)
    sfr = selected_feature_ratio(score_map, threshold=0.5)
    spr = sparsity_from_ratio(sfr)
    sample_elapsed = time.time() - sample_t0
    print(f"[{i+1:03d}/{len(test_loader):03d}] explain done | explain={t_explain:.3f}s | total={sample_elapsed:.3f}s | aopc={aopc:.4f} | auc={auci:.4f}")

    records.append({
        'idx': int(i),
        'path': rel[0],
        'y_true': int(y.item()),
        'y_pred': int(pred),
        'explain_time_sec': float(t_explain),
        'aopc_deletion': float(aopc),
        'auc_insertion': float(auci),
        'iou_top10': float(siou),
        'comprehensiveness': float(comp),
        'sufficiency': float(suff),
        'selected_feature_ratio': float(sfr),
        'sparsity': float(spr),
    })

res_df = pd.DataFrame(records)

summary = {
    'method': METHOD_KEY,
    'variant': VARIANT_KEY,
    'baseline_key': BASELINE_KEY,
    'num_samples': int(len(res_df)),
    'metrics': {
        'mean_explain_time_sec': float(res_df['explain_time_sec'].mean()) if len(res_df) else None,
        'median_explain_time_sec': float(res_df['explain_time_sec'].median()) if len(res_df) else None,
        'mean_aopc_deletion': float(res_df['aopc_deletion'].mean()) if len(res_df) else None,
        'mean_auc_insertion': float(res_df['auc_insertion'].mean()) if len(res_df) else None,
        'mean_iou_top10': float(res_df['iou_top10'].mean()) if len(res_df) else None,
        'mean_comprehensiveness': float(res_df['comprehensiveness'].mean()) if len(res_df) else None,
        'mean_sufficiency': float(res_df['sufficiency'].mean()) if len(res_df) else None,
        'mean_selected_feature_ratio': float(res_df['selected_feature_ratio'].mean()) if len(res_df) else None,
        'mean_sparsity': float(res_df['sparsity'].mean()) if len(res_df) else None,
    }
}

OUT_DIR.mkdir(parents=True, exist_ok=True)
(res_df).to_csv(OUT_DIR / 'per_sample_metrics.csv', index=False, encoding='utf-8')
(OUT_DIR / 'quant_metrics.json').write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8')
print('[OK] saved:', OUT_DIR)
print(json.dumps(summary, indent=2, ensure_ascii=False))



